In [1]:
!pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

data = fetch_ucirepo(id=967)
df = pd.concat([data.data.ids, data.data.features, data.data.targets], axis=1)

print(df.shape)
print(df.columns.tolist())

(235795, 55)
['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


In [3]:
print(df[["URL"]].head(10))
print(df["label"].value_counts())

                                  URL
0    https://www.southbankmosaics.com
1            https://www.uni-mainz.de
2      https://www.voicefmradio.co.uk
3         https://www.sfnmjournal.com
4  https://www.rewildingargentina.org
5     https://www.globalreporting.org
6          https://www.saffronart.com
7          https://www.nerdscandy.com
8      https://www.hyderabadonline.in
9                 https://www.aap.org
label
1    134850
0    100945
Name: count, dtype: int64


In [4]:
import re
from urllib.parse import urlparse

def extract_features(url):
    url = str(url).strip()
    parsed = urlparse(url if "://" in url else "http://" + url)
    host = parsed.netloc.split(":")[0].lower()
    parts = host.split(".")

    return {
        "url_length": len(url),
        "domain_length": len(host),
        "num_subdomains": max(len(parts) - 2, 0),
        "is_ip": int(bool(re.fullmatch(r"\d{1,3}(\.\d{1,3}){3}", host))),
        "is_https": int(parsed.scheme == "https"),
        "num_digits": sum(c.isdigit() for c in url),
        "num_letters": sum(c.isalpha() for c in url),
        "num_equals": url.count("="),
        "num_qmark": url.count("?"),
        "num_ampersand": url.count("&"),
        "num_dots": url.count("."),
        "num_hyphens_in_domain": host.count("-"),
        "has_at_symbol": int("@" in url),
        "num_slashes": url.count("/"),
        "path_length": len(parsed.path),
    }

print(extract_features("https://www.google.com/search?q=hello"))
print(extract_features("http://192.168.1.1/login@secure-paypal.com"))

{'url_length': 37, 'domain_length': 14, 'num_subdomains': 1, 'is_ip': 0, 'is_https': 1, 'num_digits': 0, 'num_letters': 29, 'num_equals': 1, 'num_qmark': 1, 'num_ampersand': 0, 'num_dots': 2, 'num_hyphens_in_domain': 0, 'has_at_symbol': 0, 'num_slashes': 3, 'path_length': 7}
{'url_length': 42, 'domain_length': 11, 'num_subdomains': 2, 'is_ip': 1, 'is_https': 0, 'num_digits': 8, 'num_letters': 24, 'num_equals': 0, 'num_qmark': 0, 'num_ampersand': 0, 'num_dots': 4, 'num_hyphens_in_domain': 0, 'has_at_symbol': 1, 'num_slashes': 3, 'path_length': 24}


In [5]:
features = pd.DataFrame([extract_features(u) for u in df["URL"]])
labels = (df["label"] == 0).astype(int)   # 1 = phishing, 0 = legitimate

print(features.shape)
print(labels.value_counts())

(235795, 15)
label
0    134850
1    100945
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Fix the IP quirk
features.loc[features["is_ip"] == 1, "num_subdomains"] = 0

# Remove duplicate URLs (compare on the link text, before splitting)
data_all = features.copy()
data_all["URL"] = df["URL"].values
data_all["label"] = labels.values
data_all = data_all.drop_duplicates(subset="URL")

print("Rows after removing duplicate URLs:", data_all.shape[0])
print(data_all["label"].value_counts())

X = data_all.drop(columns=["URL", "label"])
y = data_all["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)

Rows after removing duplicate URLs: 235370
label
0    134850
1    100520
Name: count, dtype: int64
Train: (188296, 15) Test: (47074, 15)


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), target_names=["legit", "phishing"]))

              precision    recall  f1-score   support

       legit       0.99      1.00      1.00     26970
    phishing       1.00      0.99      0.99     20104

    accuracy                           1.00     47074
   macro avg       1.00      1.00      1.00     47074
weighted avg       1.00      1.00      1.00     47074



In [8]:
import pandas as pd

importance = pd.Series(rf.feature_importances_, index=X_train.columns)
print(importance.sort_values(ascending=False).round(3))

is_https                 0.434
num_slashes              0.191
path_length              0.164
num_digits               0.060
url_length               0.054
domain_length            0.031
num_letters              0.021
num_subdomains           0.017
num_dots                 0.014
num_hyphens_in_domain    0.011
num_qmark                0.002
num_equals               0.000
is_ip                    0.000
has_at_symbol            0.000
num_ampersand            0.000
dtype: float64


In [9]:
check = X.copy()
check["label"] = y.values
print(check.groupby("label").mean().T.round(2))

label                      0      1
url_length             27.23  46.22
domain_length          19.23  24.44
num_subdomains          1.16   1.16
is_ip                   0.00   0.01
is_https                1.00   0.49
num_digits              0.05   4.34
num_letters            21.93  34.24
num_equals              0.00   0.15
num_qmark               0.00   0.07
num_ampersand           0.00   0.08
num_dots                2.16   2.38
num_hyphens_in_domain   0.08   0.55
has_at_symbol           0.00   0.01
num_slashes             2.00   3.02
path_length             0.00   8.71


In [10]:
print("PHISHING examples:")
print(data_all[data_all["label"] == 1]["URL"].sample(10, random_state=1).to_string(index=False))
print()
print("LEGIT examples:")
print(data_all[data_all["label"] == 0]["URL"].sample(10, random_state=1).to_string(index=False))

PHISHING examples:
  https://bancodeoccidente.servidorprivad.repl.co/
http://2001453526565.20331064515.my.id/failure.php
                   http://www.eblagh-adiiran-ir.ga
                             http://www.arkauto.ru
     https://emerald-exch-0utlook.firebaseapp.com/
                 http://www.dsme.co.kr.snocomb.com
https://williamsyahoomailboxupdate213.weeblysit...
https://broad-rain-fadf.grebbca77n0ttpmtd87o.wo...
           https://www.ekel-nat.ekinsrr.bffzwh.top
                          http://www.doctorsby.com

LEGIT examples:
            https://www.levelup.com
        https://www.notjusttoyz.com
  https://www.discover-suriname.com
      https://www.ecolabelindex.com
        https://www.cinematheque.fr
https://www.necclassicmotorshow.com
           https://www.ringling.org
       https://www.hlcommission.org
        https://www.divx-digest.com
       https://www.smartgecko.co.za


In [11]:
tests = [
    "https://www.google.com",
    "https://github.com/login",
    "https://www.bbc.com/news/world-africa-12345678",
    "https://en.wikipedia.org/wiki/Phishing",
    "https://secure-paypal-login.example.com/verify",
    "https://paypal.com.account-verify.xyz/login",
    "http://192.168.1.5/bank/update.php",
    "https://www.google.com.security-check.top/signin",
]

test_df = pd.DataFrame([extract_features(u) for u in tests])
test_df.loc[test_df["is_ip"] == 1, "num_subdomains"] = 0
test_df = test_df[X_train.columns]

probs = rf.predict_proba(test_df)[:, 1]
for u, p in zip(tests, probs):
    print(f"{p:.2f}  {u}")

0.00  https://www.google.com
1.00  https://github.com/login
1.00  https://www.bbc.com/news/world-africa-12345678
1.00  https://en.wikipedia.org/wiki/Phishing
1.00  https://secure-paypal-login.example.com/verify
1.00  https://paypal.com.account-verify.xyz/login
1.00  http://192.168.1.5/bank/update.php
1.00  https://www.google.com.security-check.top/signin


In [24]:
import re
from urllib.parse import urlparse

SUSPICIOUS_WORDS = ["login", "signin", "verify", "secure", "account", "update",
                    "bank", "confirm", "password", "wallet", "free", "bonus"]
BRANDS = ["paypal", "google", "apple", "microsoft", "amazon", "facebook",
          "netflix", "bank", "instagram", "whatsapp"]
RISKY_TLDS = {"top", "xyz", "ga", "tk", "ml", "cf", "gq", "icu", "buzz", "club", "online", "site"}

def safe_parse(url):
    try:
        return urlparse(url if "://" in url else "http://" + url)
    except ValueError:
        # bad link (for example a stray bracket): remove brackets and try again
        cleaned = url.replace("[", "").replace("]", "")
        return urlparse(cleaned if "://" in cleaned else "http://" + cleaned)

def extract_features(url):
    url = str(url).strip().lower()
    parsed = safe_parse(url)
    host = parsed.netloc.split("@")[-1].split(":")[0]
    parts = host.split(".")
    is_ip = int(bool(re.fullmatch(r"\d{1,3}(\.\d{1,3}){3}", host)))
    tld = parts[-1] if parts else ""
    main_domain = ".".join(parts[-2:]) if len(parts) >= 2 else host
    domain_name = parts[-2] if len(parts) >= 2 else host
    rest = url.replace(main_domain, "")

    return {
        "domain_length": len(domain_name),
        "num_subdomains": 0 if is_ip else max(len(parts) - 2, 0),
        "is_ip": is_ip,
        "digits_in_domain": sum(c.isdigit() for c in host),
        "hyphens_in_domain": host.count("-"),
        "has_at_symbol": int("@" in url),
        "risky_tld": int(tld in RISKY_TLDS),
        "num_suspicious_words": sum(w in url for w in SUSPICIOUS_WORDS),
        "brand_outside_domain": int(any(b in rest and b not in domain_name for b in BRANDS)),
        "brand_in_subdomain": int(any(b in ".".join(parts[:-2]) for b in BRANDS)),
        "digit_ratio_domain": sum(c.isdigit() for c in domain_name) / max(len(domain_name), 1),
        "vowel_ratio_domain": sum(c in "aeiou" for c in domain_name) / max(len(domain_name), 1),
        "has_query": int("?" in url),
        "num_dots": host.count("."),
    }

print(extract_features("https://paypal.com.account-verify.xyz/login"))
print(extract_features("http://bad[link.com/page"))

{'domain_length': 14, 'num_subdomains': 2, 'is_ip': 0, 'digits_in_domain': 0, 'hyphens_in_domain': 1, 'has_at_symbol': 0, 'risky_tld': 1, 'num_suspicious_words': 3, 'brand_outside_domain': 1, 'brand_in_subdomain': 1, 'digit_ratio_domain': 0.0, 'vowel_ratio_domain': 0.35714285714285715, 'has_query': 0, 'num_dots': 3}
{'domain_length': 7, 'num_subdomains': 0, 'is_ip': 0, 'digits_in_domain': 0, 'hyphens_in_domain': 0, 'has_at_symbol': 0, 'risky_tld': 0, 'num_suspicious_words': 0, 'brand_outside_domain': 0, 'brand_in_subdomain': 0, 'digit_ratio_domain': 0.0, 'vowel_ratio_domain': 0.2857142857142857, 'has_query': 0, 'num_dots': 1}


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

feats = pd.DataFrame([extract_features(u) for u in data_all["URL"]])
y = data_all["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    feats, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), target_names=["legit", "phishing"]))

# real-world test
tests = [
    "https://www.google.com",
    "https://github.com/login",
    "https://www.bbc.com/news/world-africa-12345678",
    "https://en.wikipedia.org/wiki/Phishing",
    "https://secure-paypal-login.example.com/verify",
    "https://paypal.com.account-verify.xyz/login",
    "http://192.168.1.5/bank/update.php",
    "https://www.google.com.security-check.top/signin",
]
test_df = pd.DataFrame([extract_features(u) for u in tests])[X_train.columns]
for u, p in zip(tests, rf.predict_proba(test_df)[:, 1]):
    print(f"{p:.2f}  {u}")

              precision    recall  f1-score   support

       legit       0.81      0.96      0.88     26970
    phishing       0.93      0.71      0.80     20104

    accuracy                           0.85     47074
   macro avg       0.87      0.83      0.84     47074
weighted avg       0.86      0.85      0.85     47074

0.32  https://www.google.com
1.00  https://github.com/login
0.11  https://www.bbc.com/news/world-africa-12345678
0.14  https://en.wikipedia.org/wiki/Phishing
0.99  https://secure-paypal-login.example.com/verify
0.97  https://paypal.com.account-verify.xyz/login
1.00  http://192.168.1.5/bank/update.php
1.00  https://www.google.com.security-check.top/signin


In [16]:
import itertools

paths = ["login", "signin", "account/settings", "verify", "secure/checkout",
         "news/article-12345", "wiki/Python", "search?q=test", "help/update-password",
         "user/profile", "docs/getting-started", "about", "contact", "products/item-99"]

extra_urls = [f"https://www.{d}/{p}" for d, p in itertools.product(TRUSTED, paths)]
extra = pd.DataFrame({"URL": extra_urls, "label": 0})

data_all2 = pd.concat([data_all[["URL", "label"]], extra], ignore_index=True)
print("New total:", len(data_all2), " added legit-with-path rows:", len(extra))

New total: 235580  added legit-with-path rows: 210


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

feats = pd.DataFrame([extract_features(u) for u in data_all2["URL"]])
y = data_all2["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    feats, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), target_names=["legit", "phishing"]))

tests = [
    "https://www.google.com",
    "https://github.com/login",
    "https://www.bbc.com/news/world-africa-12345678",
    "https://en.wikipedia.org/wiki/Phishing",
    "https://secure-paypal-login.example.com/verify",
    "https://paypal.com.account-verify.xyz/login",
    "http://192.168.1.5/bank/update.php",
    "https://www.google.com.security-check.top/signin",
]
test_df = pd.DataFrame([extract_features(u) for u in tests])[X_train.columns]
for u, p in zip(tests, rf.predict_proba(test_df)[:, 1]):
    print(f"{p:.2f}  {u}")

              precision    recall  f1-score   support

       legit       0.82      0.96      0.88     27012
    phishing       0.93      0.71      0.80     20104

    accuracy                           0.85     47116
   macro avg       0.87      0.83      0.84     47116
weighted avg       0.86      0.85      0.85     47116

0.95  https://www.google.com
0.86  https://github.com/login
0.00  https://www.bbc.com/news/world-africa-12345678
0.00  https://en.wikipedia.org/wiki/Phishing
1.00  https://secure-paypal-login.example.com/verify
0.93  https://paypal.com.account-verify.xyz/login
1.00  http://192.168.1.5/bank/update.php
1.00  https://www.google.com.security-check.top/signin


In [18]:
feats_check = feats.copy()
feats_check["label"] = y
feats_check["URL"] = data_all2["URL"].values

trusted_rows = feats_check[feats_check["is_trusted_domain"] == 1]
print("Rows with a trusted domain, by label:")
print(trusted_rows["label"].value_counts())
print()
print("Phishing-labeled links on trusted domains (first 15):")
print(trusted_rows[trusted_rows["label"] == 1]["URL"].head(15).to_string(index=False))
print()
print("Is the google.com homepage in the data?")
print(feats_check[feats_check["URL"].str.contains(r"^https?://(www\.)?google\.com/?$", regex=True)][["URL", "label"]])

Rows with a trusted domain, by label:
label
1    783
0    213
Name: count, dtype: int64

Phishing-labeled links on trusted domains (first 15):
https://docs.google.com/forms/d/1casphevkfdaz2o...
https://docs.google.com/forms/d/e/1faipqlscckv1...
https://www.google.com/url?q=https://ipfs.io/ip...
https://docs.google.com/forms/d/e/1faipqlsdnega...
https://docs.google.com/forms/d/e/1faipqlsfp1eu...
https://docs.google.com/forms/d/e/1faipqlsc-nv7...
https://drive.google.com/file/d/1ginbnlpvt7kpfn...
https://docs.google.com/presentation/d/e/2pacx-...
https://docs.google.com/forms/d/e/1faipqlseht4c...
https://sites.google.com/view/necrologieinfosfr...
https://docs.google.com/presentation/d/e/2pacx-...
https://docs.google.com/forms/d/e/1faipqlsfjsgj...
          https://sites.google.com/view/ptss18/?dn
https://storage.cloud.google.com/maintainanceco...
https://docs.google.com/presentation/d/e/2pacx-...

Is the google.com homepage in the data?


/tmp/ipykernel_2767/3317036491.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print(feats_check[feats_check["URL"].str.contains(r"^https?://(www\.)?google\.com/?$", regex=True)][["URL", "label"]])


Empty DataFrame
Columns: [URL, label]
Index: []


In [19]:
import pandas as pd

raw = pd.read_csv("malicious_phish.csv")
print(raw.shape)
print(raw["type"].value_counts())
print(raw.sample(12, random_state=1).to_string())

(211726, 2)
type
benign        154779
defacement     39170
phishing       12707
malware         5069
defaceme           1
Name: count, dtype: int64
                                                                                                                                                                                  url        type
12578                                                                                                                        southafrica.indymedia.org/print.php?id=4973&comments=yes      benign
15685                                                                                                                      billboard.com/artist/anita-pointer/discography/songs/23169      benign
79101                                                                                                                          projects.latimes.com/hollywood/star-walk/friz-freleng/      benign
62050                                                                       

In [20]:
import pandas as pd

raw = pd.read_csv("malicious_phish.csv")
print(raw.shape)
print(raw["type"].value_counts())
print(raw.sample(12, random_state=1).to_string())

(651191, 2)
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64
                                                                                                                                                             url        type
167113                                                                                                                                         apolloduck.co.za/      benign
387680                                                                                                                       acronyms.thefreedictionary.com/WDOM      benign
528900                                                                                                        https://nepan.org.np/Alibaba/Alibaba.com/Login.htm    phishing
251286                                                                                       soundonsound.com/sos/aug06/articles/rogernichols_0806.htm?print=yes      benign
433650  

In [21]:
data3 = raw[raw["type"].isin(["benign", "phishing"])].copy()
data3["label"] = (data3["type"] == "phishing").astype(int)
data3 = data3.drop_duplicates(subset="url")
print("Rows after filtering and removing duplicates:", len(data3))
print(data3["label"].value_counts())

# Look for shortcuts
data3["has_scheme"] = data3["url"].str.startswith(("http://", "https://")).astype(int)
data3["has_path"] = data3["url"].str.replace(r"^https?://", "", regex=True).str.contains("/").astype(int)
data3["length"] = data3["url"].str.len()
print()
print(data3.groupby("label")[["has_scheme", "has_path", "length"]].mean().round(2))

Rows after filtering and removing duplicates: 522166
label
0    428080
1     94086
Name: count, dtype: int64

       has_scheme  has_path  length
label                              
0            0.08      1.00   57.68
1            0.26      0.77   45.85


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

feats3 = pd.DataFrame([extract_features(u) for u in data3["url"]])
y3 = data3["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    feats3, y3, test_size=0.2, random_state=42, stratify=y3
)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                            class_weight="balanced_subsample")
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), target_names=["legit", "phishing"]))

tests = [
    "https://www.google.com",
    "https://github.com/login",
    "https://www.bbc.com/news/world-africa-12345678",
    "https://en.wikipedia.org/wiki/Phishing",
    "https://stackoverflow.com/questions/11227809/how-to",
    "https://www.nytimes.com/section/world",
    "https://secure-paypal-login.example.com/verify",
    "https://paypal.com.account-verify.xyz/login",
    "http://192.168.1.5/bank/update.php",
    "https://www.google.com.security-check.top/signin",
    "https://www.bbc.com",
    "https://www.bbc.com/",
]
test_df = pd.DataFrame([extract_features(u) for u in tests])[X_train.columns]
for u, p in zip(tests, rf.predict_proba(test_df)[:, 1]):
    print(f"{p:.2f}  {u}")

              precision    recall  f1-score   support

       legit       0.94      0.82      0.88     85617
    phishing       0.48      0.74      0.58     18817

    accuracy                           0.81    104434
   macro avg       0.71      0.78      0.73    104434
weighted avg       0.85      0.81      0.82    104434

0.83  https://www.google.com
0.63  https://github.com/login
0.74  https://www.bbc.com/news/world-africa-12345678
0.19  https://en.wikipedia.org/wiki/Phishing
0.24  https://stackoverflow.com/questions/11227809/how-to
0.69  https://www.nytimes.com/section/world
0.88  https://secure-paypal-login.example.com/verify
0.94  https://paypal.com.account-verify.xyz/login
1.00  http://192.168.1.5/bank/update.php
0.73  https://www.google.com.security-check.top/signin
0.74  https://www.bbc.com
0.74  https://www.bbc.com/


In [26]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def normalize(url):
    u = str(url).strip().lower()
    u = re.sub(r"^[a-z]+://", "", u)   # remove http:// or https://
    u = re.sub(r"^www\.", "", u)       # remove leading www.
    u = u.rstrip("/")                  # remove trailing slash
    return u

data3["clean"] = data3["url"].map(normalize)

Xtr_txt, Xte_txt, ytr, yte = train_test_split(
    data3["clean"], data3["label"].values,
    test_size=0.2, random_state=42, stratify=data3["label"].values
)

vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 5),
                      max_features=300000, sublinear_tf=True)
Xtr = vec.fit_transform(Xtr_txt)
Xte = vec.transform(Xte_txt)

clf = SGDClassifier(loss="log_loss", class_weight="balanced",
                    max_iter=30, random_state=42)
clf.fit(Xtr, ytr)
print(classification_report(yte, clf.predict(Xte), target_names=["legit", "phishing"]))

tests = [
    "https://www.google.com",
    "https://github.com/login",
    "https://www.bbc.com/news/world-africa-12345678",
    "https://en.wikipedia.org/wiki/Phishing",
    "https://stackoverflow.com/questions/11227809/how-to",
    "https://www.nytimes.com/section/world",
    "https://secure-paypal-login.example.com/verify",
    "https://paypal.com.account-verify.xyz/login",
    "http://192.168.1.5/bank/update.php",
    "https://www.google.com.security-check.top/signin",
]
probs = clf.predict_proba(vec.transform([normalize(u) for u in tests]))[:, 1]
for u, p in zip(tests, probs):
    print(f"{p:.2f}  {u}")

              precision    recall  f1-score   support

       legit       0.97      0.83      0.89     85617
    phishing       0.53      0.88      0.66     18817

    accuracy                           0.84    104434
   macro avg       0.75      0.85      0.78    104434
weighted avg       0.89      0.84      0.85    104434

0.84  https://www.google.com
0.61  https://github.com/login
0.34  https://www.bbc.com/news/world-africa-12345678
0.19  https://en.wikipedia.org/wiki/Phishing
0.25  https://stackoverflow.com/questions/11227809/how-to
0.34  https://www.nytimes.com/section/world
0.63  https://secure-paypal-login.example.com/verify
0.87  https://paypal.com.account-verify.xyz/login
0.72  http://192.168.1.5/bank/update.php
0.88  https://www.google.com.security-check.top/signin


In [27]:
bare = data3["clean"].map(lambda u: "/" not in u and "?" not in u)
print("Bare domains (no path), by label:")
print(data3[bare]["label"].value_counts())
print()
print("Share of bare domains that are phishing:", round(data3[bare]["label"].mean(), 2))
print("Share of ALL links that are phishing:  ", round(data3["label"].mean(), 2))
print()
g = data3["clean"].str.contains("google.com", regex=False)
print("Links containing google.com, by label:")
print(data3[g]["label"].value_counts())

Bare domains (no path), by label:
label
0    47606
1    23537
Name: count, dtype: int64

Share of bare domains that are phishing: 0.33
Share of ALL links that are phishing:   0.18

Links containing google.com, by label:
label
0    977
1    720
Name: count, dtype: int64


In [28]:
from sklearn.metrics import precision_score, recall_score

probs_te = clf.predict_proba(Xte)[:, 1]
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    pred = (probs_te >= t).astype(int)
    print(f"cutoff {t}: precision {precision_score(yte, pred):.2f}, recall {recall_score(yte, pred):.2f}")

cutoff 0.5: precision 0.53, recall 0.88
cutoff 0.6: precision 0.67, recall 0.69
cutoff 0.7: precision 0.79, recall 0.42
cutoff 0.8: precision 0.91, recall 0.19
cutoff 0.9: precision 0.99, recall 0.07


In [29]:
gl = data3[data3["clean"].str.contains("google.com", regex=False)]
print("LEGIT links containing google.com:")
print(gl[gl["label"] == 0]["clean"].sample(10, random_state=1).to_string(index=False))
print()
print("PHISHING links containing google.com:")
print(gl[gl["label"] == 1]["clean"].sample(10, random_state=1).to_string(index=False))
print()
print("Exact 'google.com' rows in the data:", (data3["clean"] == "google.com").sum())

LEGIT links containing google.com:
docs.google.com/spreadsheet/viewform?formkey=dh...
spreadsheets.google.com/spreadsheet/formrespons...
books.google.com/books/about/municipal_housecle...
code.google.com/apis/chart/interactive/docs/gal...
sites.google.com/site/grstenliuktla/ohio-genera...
groups.google.com/group/android-porting/browse_...
google.com.au/url?sa=t&amp;rct=j&amp;q=site:sui...
books.google.com/books/about/marguerite_bourgeo...
docs.google.com/a/goliadcountytx.gov/spreadshee...
google.com/hostednews/afp/article/aleqm5je08l64...

PHISHING links containing google.com:
docs.google.com/spreadsheet/viewform?formkey=df...
spreadsheets.google.com/spreadsheet/viewform?hl...
             sites.google.com/view/embedded-f/home
                      code.google.com/p/cryptsetup
docs.google.com/a/nebo.edu/spreadsheet/viewform...
spreadsheets.google.com/spreadsheet/embeddedfor...
    sites.google.com/site/importautostocanada/home
docs.google.com/forms/d/e/1faipqlsdpu1i7oofsplo...
google.c

In [30]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import precision_score, recall_score, classification_report

MAX_LEN = 100

def encode(urls):
    arr = np.zeros((len(urls), MAX_LEN), dtype="int32")
    for i, u in enumerate(urls):
        codes = [min(ord(c), 127) + 1 for c in u[:MAX_LEN]]
        arr[i, :len(codes)] = codes
    return arr

Xtr_enc = encode(list(Xtr_txt))
Xte_enc = encode(list(Xte_txt))

model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(129, 32),
    layers.Conv1D(128, 5, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                           restore_best_weights=True)
model.fit(Xtr_enc, ytr, validation_split=0.1, epochs=8,
          batch_size=256, callbacks=[early_stop])

probs_k = model.predict(Xte_enc).ravel()
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (probs_k >= t).astype(int)
    print(f"cutoff {t}: precision {precision_score(yte, pred):.2f}, recall {recall_score(yte, pred):.2f}")

tests = [
    "https://www.google.com",
    "https://github.com/login",
    "https://www.bbc.com/news/world-africa-12345678",
    "https://en.wikipedia.org/wiki/Phishing",
    "https://stackoverflow.com/questions/11227809/how-to",
    "https://www.nytimes.com/section/world",
    "https://secure-paypal-login.example.com/verify",
    "https://paypal.com.account-verify.xyz/login",
    "http://192.168.1.5/bank/update.php",
    "https://www.google.com.security-check.top/signin",
]
tp = model.predict(encode([normalize(u) for u in tests])).ravel()
for u, p in zip(tests, tp):
    print(f"{p:.2f}  {u}")

Epoch 1/8
1469/1469 ━━━━━━━━━━━━━━━━━━━━ 153s 103ms/step - accuracy: 0.8672 - loss: 0.3031 - val_accuracy: 0.8906 - val_loss: 0.2531
Epoch 2/8
1469/1469 ━━━━━━━━━━━━━━━━━━━━ 148s 101ms/step - accuracy: 0.8965 - loss: 0.2386 - val_accuracy: 0.9007 - val_loss: 0.2276
Epoch 3/8
1469/1469 ━━━━━━━━━━━━━━━━━━━━ 204s 103ms/step - accuracy: 0.9043 - loss: 0.2211 - val_accuracy: 0.8987 - val_loss: 0.2381
Epoch 4/8
1469/1469 ━━━━━━━━━━━━━━━━━━━━ 200s 101ms/step - accuracy: 0.9091 - loss: 0.2113 - val_accuracy: 0.9051 - val_loss: 0.2210
Epoch 5/8
1469/1469 ━━━━━━━━━━━━━━━━━━━━ 148s 101ms/step - accuracy: 0.9123 - loss: 0.2040 - val_accuracy: 0.9089 - val_loss: 0.2149
Epoch 6/8
1469/1469 ━━━━━━━━━━━━━━━━━━━━ 203s 102ms/step - accuracy: 0.9146 - loss: 0.1990 - val_accuracy: 0.9079 - val_loss: 0.2152
Epoch 7/8
1469/1469 ━━━━━━━━━━━━━━━━━━━━ 149s 102ms/step - accuracy: 0.9169 - loss: 0.1939 - val_accuracy: 0.9075 - val_loss: 0.2248
3264/3264 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step
cutoff 0.3: precision 0.6

In [31]:
import json

model.save("phishing_char_model.keras")
with open("config.json", "w") as f:
    json.dump({"max_len": MAX_LEN, "threshold": 0.4}, f)
print("Saved")

Saved


In [32]:
code = '''import re
import numpy as np

MAX_LEN = 100

def normalize(url):
    u = str(url).strip().lower()
    u = re.sub(r"^[a-z]+://", "", u)
    u = re.sub(r"^www\\.", "", u)
    u = u.rstrip("/")
    return u

def encode(urls):
    arr = np.zeros((len(urls), MAX_LEN), dtype="int32")
    for i, u in enumerate(urls):
        codes = [min(ord(c), 127) + 1 for c in u[:MAX_LEN]]
        arr[i, :len(codes)] = codes
    return arr
'''
with open("preprocess.py", "w") as f:
    f.write(code)
print(open("preprocess.py").read())

import re
import numpy as np

MAX_LEN = 100

def normalize(url):
    u = str(url).strip().lower()
    u = re.sub(r"^[a-z]+://", "", u)
    u = re.sub(r"^www\.", "", u)
    u = u.rstrip("/")
    return u

def encode(urls):
    arr = np.zeros((len(urls), MAX_LEN), dtype="int32")
    for i, u in enumerate(urls):
        codes = [min(ord(c), 127) + 1 for c in u[:MAX_LEN]]
        arr[i, :len(codes)] = codes
    return arr

